# Parametric source spectra

Besides Pickles spectral types and on-disk spectra, `get_scene(name=...)` builds a source
spectrum on the fly from four **parametric** types:

| `name` | params | meaning |
|---|---|---|
| `blackbody` | `teff` (K) | Planck spectrum, normalized to `mag` |
| `flat` | `flat_unit='fnu'` (default, AB-flat) or `'flam'` | constant F_nu or F_lambda |
| `powerlaw` | `alpha`, `lambda_ref=5500` | F_lambda ∝ (λ/λ_ref)^α |
| `emission` | `lines=[{wave, flux, fwhm}]`, `mag=None` | Gaussian lines, **absolute** flux |

Every source plots itself with `scene.source.show(...)`, which takes a wavelength grid and
a flux unit — `flux_unit='flam'` gives erg/s/cm²/Å.

These spectra are validated against analytic physics (Planck, Wien, the F_nu/F_lambda
relation, Gaussian profiles, Pogson scaling) in `tests/scene/test_source_physics.py`; this
notebook is the tour, not the proof.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import wcc_etc

wcc_etc.set_wcc_style()

BKG = {"bandpass": "johnson_r", "mag": 22.5}
WAVE = np.arange(3500, 9500, 5.0)  # optical plotting grid [Angstrom]


def source(name, **kw):
    """The source element of a scene of the given type, on a zodi background."""
    kw.setdefault("bandpass", "johnson_r")
    return wcc_etc.get_scene(
        name=name, background="zodi", background_prop=BKG, **kw
    ).source

## 1. Blackbody

`name='blackbody', teff=<K>` builds a Planck spectrum normalized to the requested
broadband magnitude. Dashed lines mark Wien's displacement peak, λ_max = 2.8977719e7 / T Å.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for teff in (3000, 5500, 9000):
    src = source("blackbody", teff=teff, mag=15)
    src.show(ax=ax, wave=WAVE, flux_unit="flam", label=f"{teff} K")
    ax.axvline(
        2.8977719e7 / teff, color=ax.lines[-1].get_color(), ls="--", lw=1, alpha=0.6
    )

ax.set_yscale("log")
ax.set_title("Blackbodies at r = 15, with Wien peaks (dashed)")
ax.legend(fontsize=9)
plt.show()

## 2. Flat

`name='flat'` defaults to `flat_unit='fnu'` — flat in F_nu, the AB reference, which falls
as λ⁻² in F_lambda. `flat_unit='flam'` is flat in F_lambda instead.

In [ ]:
fig, (ax_fnu, ax_flam) = plt.subplots(1, 2, figsize=(11, 4))
for unit in ("fnu", "flam"):
    src = source("flat", flat_unit=unit, mag=15)
    src.show(ax=ax_fnu, wave=WAVE, flux_unit="fnu", label=f"flat_unit='{unit}'")
    src.show(ax=ax_flam, wave=WAVE, flux_unit="flam", label=f"flat_unit='{unit}'")

ax_fnu.set_title("Seen in F_nu")
ax_flam.set_title("Seen in F_lambda")
for ax in (ax_fnu, ax_flam):
    ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 3. Power law

`name='powerlaw', alpha=<exponent>` builds F_lambda ∝ (λ/λ_ref)^α — the public `alpha` is
the *F_lambda* exponent, so `alpha=0` is flat in F_lambda, negative is red, positive blue.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for alpha in (-2.0, -1.0, 0.0, 1.0, 2.0):
    source("powerlaw", alpha=alpha, mag=15).show(
        ax=ax, wave=WAVE, flux_unit="flam", label=f"alpha = {alpha:+.0f}"
    )

ax.set_yscale("log")
ax.set_title("Power-law sources at r = 15")
ax.legend(fontsize=9)
plt.show()

## 4. Emission lines

`name='emission', mag=None, lines=[{wave, flux, fwhm}]` sums Gaussian profiles where
`flux` is the **absolute** integrated line flux in erg/s/cm² and `fwhm` defaults to 2 Å.
With `mag=None` the source carries absolute flux and is not broadband-normalized.

In [ ]:
src = source(
    "emission",
    mag=None,
    lines=[
        {"wave": 6563, "flux": 1e-15, "fwhm": 4},  # Halpha
        {"wave": 6583, "flux": 4e-16, "fwhm": 4},  # [NII]
    ],
)

fig, ax = plt.subplots(figsize=(8, 4.5))
src.show(ax=ax, wave=np.arange(6500, 6650, 0.05), flux_unit="flam")
ax.set_title("Halpha 6563 + [NII] 6583, FWHM 4 A, flux ratio 2.5")
plt.show()

## 5. Driving the ETC

Each continuum source feeds the full `Simulation` through the broadband `sony:r` filter.
The SNR follows Pogson scaling — five magnitudes fainter is a hundredth of the count rate.
(Narrowband filters have no throughput curves yet, so emission sources cannot be observed
through them.)

In [ ]:
SPECS = {
    "blackbody 5500 K": ("blackbody", {"teff": 5500}),
    "flat (AB)": ("flat", {}),
    "powerlaw alpha = -1": ("powerlaw", {"alpha": -1.0}),
}
mags = np.linspace(18, 25, 8)

fig, ax = plt.subplots(figsize=(8, 4.5))
for label, (name, extra) in SPECS.items():
    snr = []
    for mag in mags:
        scene = wcc_etc.get_scene(
            name=name,
            mag=mag,
            background="zodi",
            bandpass="johnson_r",
            background_prop=BKG,
            **extra,
        )
        sim = wcc_etc.Simulation.from_sensor_and_scene("sony:r", scene)
        snr.append(float(sim.get_snr(60)["snr"]))
    ax.plot(mags, snr, "o-", ms=4, label=label)

ax.set_yscale("log")
ax.set_xlabel("Source magnitude (johnson_r)")
ax.set_ylabel("SNR at 60 s")
ax.set_title("ETC SNR vs magnitude by source type")
ax.legend(fontsize=9)
plt.show()

## 6. `update()` rebuilds the spectrum

Type-appropriate **shape** parameters change in place and rebuild the spectrum. The
spectrum **type** is locked: `sim.update(source__spectrum=...)` is ignored with a warning.

At a fixed r = 19 normalization, dropping the blackbody from 8000 K to 3000 K takes the
60 s SNR from **202.4 to 155.7** — the same broadband magnitude, but a redder spectrum
places less of its light inside the `sony:r` passband.

In [ ]:
scene = wcc_etc.get_scene(
    name="blackbody",
    teff=8000,
    mag=19,
    background="zodi",
    bandpass="johnson_r",
    background_prop=BKG,
)
sim = wcc_etc.Simulation.from_sensor_and_scene("sony:r", scene)

print(f"SNR at teff = 8000 K : {sim.get_snr(60)['snr']:.1f}")
sim.update(source__teff=3000)  # cooler blackbody -> less r-band flux
print(f"SNR at teff = 3000 K : {sim.get_snr(60)['snr']:.1f}")

## 7. Normalization bandpass

`bandpass` sets the filter the source magnitude is anchored in. Alongside synphot's
built-in systems the ETC ships the SDSS primed filters as local curves — `sdss_u`,
`sdss_g`, `sdss_r`, `sdss_i`, `sdss_z` — accepted anywhere a bandpass name is.
`plot_bandpass_mpl` draws any of them; the same blackbody normalized to the same AB
magnitude in different bands produces a different continuum.

In [ ]:
from wcc_etc.io import resolve_bandpass

fig, (ax_bp, ax_norm) = plt.subplots(1, 2, figsize=(11, 4.2))

for name in ["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"]:
    wcc_etc.plot_bandpass_mpl(resolve_bandpass(name), ax=ax_bp, label=name)
ax_bp.set_title("SDSS filter curves")
ax_bp.legend(fontsize=8)

for name in ["sdss_g", "sdss_r", "sdss_i"]:
    source("blackbody", teff=5500, mag=15, magsys="abmag", bandpass=name).show(
        ax=ax_norm,
        wave=np.arange(3000, 11000, 5.0),
        flux_unit="flam",
        label=f"normalized in {name}",
    )
ax_norm.set_title("5500 K blackbody, mag = 15 AB")
ax_norm.legend(fontsize=8)

fig.tight_layout()
plt.show()

## Summary

- **blackbody** — Planck spectrum normalized to `mag`; the peak follows Wien's law.
- **flat** — AB-flat (`fnu`) falls as λ⁻² in F_lambda; `flam` is constant in F_lambda.
- **powerlaw** — F_lambda follows (λ/λ_ref)^α, with `alpha` the F_lambda exponent.
- **emission** — Gaussian lines carrying absolute integrated flux, with `mag=None`.
- All of them drive the full ETC, and `update()` rebuilds shape parameters in place while
  the spectrum type stays locked.
- `bandpass` anchors the magnitude — Johnson/Bessel/Cousins, or the local SDSS curves.

The analytic validation of each of these lives in `tests/scene/test_source_physics.py`.